# ESZA019 — Visão Computacional  
## Laboratório 4 — Calibração de Câmeras

**Autores:** Cesar de Jesus; Mariana Chiara; Vinicius de Marchi  
**Data de realização dos experimentos:** 01/07/2026 e 05/07/2026  
**Data de publicação:** 06/07/2026  
**Disciplina:** ESZA019 — Visão Computacional  
**Docente:** Prof. Celso Setsuo Kurashima

---

## Resumo

Este relatório apresenta o estudo e a implementação de um procedimento de calibração de câmeras com a biblioteca OpenCV. Foram utilizados alvos planos do tipo tabuleiro de xadrez para estimar parâmetros intrínsecos, coeficientes de distorção e parâmetros extrínsecos de duas câmeras distintas. A primeira calibração utilizou 17 imagens válidas, enquanto a segunda utilizou 15 imagens válidas. Também foram implementados dois métodos de correção de distorção: `cv2.undistort()` e a combinação `cv2.initUndistortRectifyMap()` com `cv2.remap()`.

Os resultados mostraram distâncias focais semelhantes entre as câmeras, mas diferenças relevantes no ponto principal, nos coeficientes de distorção e nos erros de calibração. A primeira câmera apresentou menor erro RMS e menor erro médio de reprojeção, indicando um ajuste mais consistente ao modelo adotado.

## 1. Introdução

Uma câmera digital transforma pontos tridimensionais da cena em coordenadas bidimensionais na imagem. No modelo pinhole, essa projeção é descrita pela geometria de perspectiva. Entretanto, câmeras reais possuem lentes e desalinhamentos que provocam distorções, especialmente próximas às bordas.

A calibração de câmera estima os parâmetros necessários para representar matematicamente esse processo. Os parâmetros intrínsecos descrevem propriedades internas da câmera, como distância focal e ponto principal. Os parâmetros extrínsecos descrevem a pose do alvo de calibração em relação à câmera. Os coeficientes de distorção modelam, principalmente, distorções radiais e tangenciais.

Neste laboratório, a calibração foi realizada com um tabuleiro de xadrez. As correspondências entre os pontos tridimensionais conhecidos do padrão e os cantos detectados nas imagens permitiram estimar os parâmetros pelo método implementado em `cv2.calibrateCamera()`.

## 2. Objetivos

- Compreender o significado dos parâmetros intrínsecos e extrínsecos.
- Calibrar duas câmeras diferentes com um alvo plano.
- Determinar a matriz intrínseca \(K\), os coeficientes de distorção, as matrizes \(R\) e os vetores \(t\).
- Avaliar a calibração por meio do erro RMS e do erro médio de reprojeção.
- Comparar os resultados obtidos para as duas câmeras.
- Corrigir distorções de imagens com `undistort()` e `remap()`.

## 3. Fundamentação teórica

### 3.1 Modelo de câmera

Um ponto no sistema de coordenadas do mundo pode ser transformado para o sistema da câmera e, posteriormente, projetado no plano da imagem. Em coordenadas homogêneas:

\[
s
\begin{bmatrix}
u\\v\\1
\end{bmatrix}
=
K
\begin{bmatrix}
R & t
\end{bmatrix}
\begin{bmatrix}
X\\Y\\Z\\1
\end{bmatrix}.
\]

A matriz intrínseca é:

\[
K=
\begin{bmatrix}
f_x & \gamma & c_x\\
0 & f_y & c_y\\
0 & 0 & 1
\end{bmatrix},
\]

em que \(f_x\) e \(f_y\) são as distâncias focais em pixels, \(\gamma\) é o skew e \((c_x,c_y)\) é o ponto principal.

### 3.2 Parâmetros extrínsecos

A matriz \(R\) descreve a orientação do tabuleiro em relação ao sistema de coordenadas da câmera. O vetor \(t\) descreve sua posição. Como o tabuleiro ocupou poses diferentes em cada fotografia, cada imagem possui seu próprio par \((R_i,t_i)\).

### 3.3 Distorção da lente

O vetor utilizado pelo OpenCV foi:

\[
dist=[k_1,k_2,p_1,p_2,k_3],
\]

em que \(k_1,k_2,k_3\) modelam a distorção radial e \(p_1,p_2\) modelam a distorção tangencial.

## 4. Materiais e procedimentos experimentais

Foram utilizados:

- duas câmeras distintas;
- computador com Python, NumPy e OpenCV;
- padrão de calibração com 7 × 9 quadrados;
- configuração de 6 × 8 cantos internos;
- imagens com resolução de 640 × 480 pixels;
- scripts de captura e calibração.

O vetor de pontos do objeto foi construído no plano \(Z=0\), adotando uma unidade por quadrado. Consequentemente, os vetores de translação estão expressos em unidades do lado do quadrado. Para convertê-los para centímetros, seria necessário multiplicar as coordenadas pelo tamanho físico de cada quadrado.

### 4.1 Aquisição e detecção de cantos

As imagens foram capturadas com o tabuleiro inteiro, variando distância, posição e inclinação. A detecção utilizou:

```python
CHECKERBOARD = (6, 8)
ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, flags)
corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
```

O refinamento subpixel aumenta a precisão das coordenadas dos cantos.

## 5. Parte 2A — calibração com as imagens de exemplo

O roteiro solicita a execução do programa com o conjunto de imagens fornecido pelo professor. Esse conjunto específico não estava entre os arquivos recebidos para a montagem deste notebook.



In [ ]:
from pathlib import Path
import glob
import cv2
import numpy as np

def calibrar_pasta(padrao_arquivos, checkerboard=(6, 8)):
    imagens = sorted(glob.glob(padrao_arquivos))
    if not imagens:
        print("Nenhuma imagem encontrada para:", padrao_arquivos)
        return None

    criterios = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        30,
        0.001,
    )

    objp = np.zeros(
        (checkerboard[0] * checkerboard[1], 3),
        np.float32,
    )
    objp[:, :2] = np.mgrid[
        0:checkerboard[0],
        0:checkerboard[1],
    ].T.reshape(-1, 2)

    objpoints = []
    imgpoints = []
    tamanho = None

    flags = (
        cv2.CALIB_CB_ADAPTIVE_THRESH
        + cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    for arquivo in imagens:
        imagem = cv2.imread(arquivo)
        if imagem is None:
            continue

        cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
        tamanho = cinza.shape[::-1]

        ret, corners = cv2.findChessboardCorners(
            cinza,
            checkerboard,
            flags,
        )

        if ret:
            corners2 = cv2.cornerSubPix(
                cinza,
                corners,
                (11, 11),
                (-1, -1),
                criterios,
            )
            objpoints.append(objp.copy())
            imgpoints.append(corners2)

    if len(objpoints) < 5:
        print("Número insuficiente de imagens válidas:", len(objpoints))
        return None

    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints,
        imgpoints,
        tamanho,
        None,
        None,
    )

    print("Imagens válidas:", len(objpoints))
    print("RMS:", rms)
    print("K:\n", K)
    print("dist:\n", dist)

    return {
        "rms": rms,
        "K": K,
        "dist": dist,
        "rvecs": rvecs,
        "tvecs": tvecs,
    }

resultado_exemplo = calibrar_pasta(
    "Amostras_Professor/*.jpg",
    checkerboard=(6, 8),
)

## 6. Parte 2B — calibração da primeira câmera

Foram analisadas e utilizadas **17 imagens**. Todos os 48 cantos internos foram encontrados.

### 6.1 Matriz intrínseca

\[
K_1=
\begin{bmatrix}
707.631628 & 0 & 324.231957\\
0 & 708.168665 & 204.995124\\
0 & 0 & 1
\end{bmatrix}.
\]

### 6.2 Coeficientes de distorção

\[
dist_1=
[0.04103063,\,
-0.15245383,\,
-0.00496125,\,
0.00096070,\,
0.43967777].
\]

### 6.3 Parâmetros numéricos

| Parâmetro | Resultado |
|---|---:|
| \(f_x\) | 707.631628 px |
| \(f_y\) | 708.168665 px |
| Aspect ratio \(f_x/f_y\) | 0.999241654 |
| Skew \(\gamma\) | 0.0 |
| Ponto principal | (324.231957, 204.995124) |
| Erro RMS | 0.196803915 |
| Erro médio de reprojeção | 0.027607273 |

Os valores de \(f_x\) e \(f_y\) são muito próximos, indicando escalas semelhantes nos dois eixos. O aspect ratio próximo de 1 é coerente com pixels aproximadamente quadrados. O skew nulo indica que o modelo não estimou inclinação entre os eixos do sensor.

In [ ]:
import numpy as np
import cv2

dados_cam1 = np.load("parametros_calibracao.npz", allow_pickle=True)

K_cam1 = dados_cam1["camera_matrix"]
dist_cam1 = dados_cam1["dist_coeffs"]
rvecs_cam1 = dados_cam1["rvecs"]
tvecs_cam1 = dados_cam1["tvecs"]

print("Matriz K da câmera 1:\n", K_cam1)
print("\nCoeficientes de distorção:\n", dist_cam1)

for i, (rvec, tvec) in enumerate(zip(rvecs_cam1, tvecs_cam1)):
    R, _ = cv2.Rodrigues(rvec)
    print(f"\nImagem {i}")
    print("Matriz R:")
    print(R)
    print("Vetor t:")
    print(tvec)

### 6.4 Significado dos conjuntos \(R\) e \(t\)

A câmera possui uma única matriz intrínseca porque seus parâmetros internos permaneceram fixos. Em contrapartida, o tabuleiro foi movimentado entre as capturas. Cada imagem representa uma relação espacial diferente entre o sistema de coordenadas do tabuleiro e o sistema da câmera. Por isso, a calibração produz um conjunto \(R_i,t_i\) para cada imagem válida.

## 7. Parte 2C — calibração da segunda câmera

Foram analisadas e utilizadas **15 imagens**, todas com resolução de 640 × 480 pixels.

### 7.1 Matriz intrínseca

\[
K_2=
\begin{bmatrix}
706.627377 & 0 & 352.749043\\
0 & 707.202948 & 255.534228\\
0 & 0 & 1
\end{bmatrix}.
\]

### 7.2 Coeficientes de distorção

\[
dist_2=
[0.57610104,\,
-5.52882622,\,
0.02302616,\,
0.02074245,\,
20.18273404].
\]

### 7.3 Parâmetros numéricos

| Parâmetro | Resultado |
|---|---:|
| \(f_x\) | 706.627377 px |
| \(f_y\) | 707.202948 px |
| Aspect ratio \(f_x/f_y\) | 0.999186131 |
| Skew \(\gamma\) | 0.0 |
| Ponto principal | (352.749043, 255.534228) |
| Erro RMS | 1.033853065 |
| Erro médio de reprojeção | 0.135895109 |

In [ ]:
import numpy as np

dados_cam2 = np.load(
    "cam2_parametros_calibracao.npz",
    allow_pickle=True,
)

print("Matriz K da câmera 2:")
print(dados_cam2["camera_matrix"])

print("\nCoeficientes de distorção:")
print(dados_cam2["dist_coeffs"])

matrizes_R_cam2 = dados_cam2["rotation_matrices"]
tvecs_cam2 = dados_cam2["tvecs"]
nomes_cam2 = dados_cam2["nomes_imagens"]

for i, (nome, R, t) in enumerate(
    zip(nomes_cam2, matrizes_R_cam2, tvecs_cam2)
):
    print(f"\nImagem {i}: {nome}")
    print("Matriz R:")
    print(R)
    print("Vetor t:")
    print(t)

## 8. Comparação entre as câmeras

| Parâmetro | Câmera 1 | Câmera 2 |
|---|---:|---:|
| \(f_x\) (px) | 707.6316 | 706.6274 |
| \(f_y\) (px) | 708.1687 | 707.2029 |
| \(c_x\) (px) | 324.2320 | 352.7490 |
| \(c_y\) (px) | 204.9951 | 255.5342 |
| Aspect ratio | 0.999242 | 0.999186 |
| Skew | 0.0 | 0.0 |
| RMS | 0.196804 | 1.033853 |
| Erro médio de reprojeção | 0.027607 | 0.135895 |

As distâncias focais são próximas, com diferença relativa pequena. Entretanto, o ponto principal da segunda câmera ficou deslocado em relação ao da primeira:

\[
\Delta c_x=28.5171	ext{ px},
\qquad
\Delta c_y=50.5391	ext{ px}.
\]

A segunda câmera apresentou erro RMS aproximadamente 5.25 vezes maior e erro médio de reprojeção aproximadamente 4.92 vezes maior. Assim, a primeira calibração apresentou melhor aderência entre os cantos observados e os pontos reprojetados.

Os coeficientes radiais da segunda câmera possuem magnitudes elevadas, especialmente \(k_2\) e \(k_3\). Isso pode representar uma lente mais distorcida, mas também pode indicar maior instabilidade numérica ou ajuste excessivo. Entre as possíveis causas estão distribuição insuficiente do tabuleiro nas bordas, desfoque, reflexos e menor precisão da câmera.

## 9. Parte 2D — correção de distorção

O roteiro exige a utilização de:

1. `cv2.getOptimalNewCameraMatrix()`;
2. `cv2.undistort()`;
3. `cv2.initUndistortRectifyMap()` e `cv2.remap()`.

Para a câmera 1 foi utilizada a imagem colorida `cam1_imagem_colorida.jpg`. Para a câmera 2, o notebook procura automaticamente, nesta ordem:

- `cam2_imagem_colorida.jpg`;
- `cam2_imagem_original_calibracao.jpg`;
- `Capturas_2/cam2_frm0.jpg`.

O arquivo escolhido para a câmera 2 deve ter sido capturado pela mesma câmera calibrada no item 2C e com a mesma resolução.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def primeiro_existente(candidatos):
    for caminho in candidatos:
        caminho = Path(caminho)
        if caminho.exists():
            return caminho
    return None

def corrigir_distorcao(
    nome_camera,
    caminho_parametros,
    caminho_imagem,
    prefixo,
    alpha=1.0,
):
    if caminho_imagem is None:
        print(f"Imagem da {nome_camera} não encontrada.")
        return None

    dados = np.load(caminho_parametros, allow_pickle=True)
    K = np.asarray(dados["camera_matrix"], dtype=np.float64)
    dist = np.asarray(dados["dist_coeffs"], dtype=np.float64)

    original = cv2.imread(str(caminho_imagem))
    if original is None:
        print("Não foi possível abrir:", caminho_imagem)
        return None

    h, w = original.shape[:2]
    tamanho = (w, h)

    nova_K, roi = cv2.getOptimalNewCameraMatrix(
        K,
        dist,
        tamanho,
        alpha,
        tamanho,
    )

    corrigida_undistort = cv2.undistort(
        original,
        K,
        dist,
        None,
        nova_K,
    )

    mapa_x, mapa_y = cv2.initUndistortRectifyMap(
        K,
        dist,
        None,
        nova_K,
        tamanho,
        cv2.CV_32FC1,
    )

    corrigida_remap = cv2.remap(
        original,
        mapa_x,
        mapa_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )

    diferenca = cv2.absdiff(
        corrigida_undistort,
        corrigida_remap,
    )

    cv2.imwrite(f"{prefixo}_original.jpg", original)
    cv2.imwrite(
        f"{prefixo}_corrigida_undistort.jpg",
        corrigida_undistort,
    )
    cv2.imwrite(
        f"{prefixo}_corrigida_remap.jpg",
        corrigida_remap,
    )
    cv2.imwrite(
        f"{prefixo}_diferenca_metodos.jpg",
        diferenca,
    )

    imagens = [
        cv2.cvtColor(original, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(corrigida_undistort, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(corrigida_remap, cv2.COLOR_BGR2RGB),
    ]

    titulos = [
        f"{nome_camera}: original",
        f"{nome_camera}: undistort",
        f"{nome_camera}: remap",
    ]

    fig, eixos = plt.subplots(1, 3, figsize=(18, 6))
    for eixo, imagem, titulo in zip(eixos, imagens, titulos):
        eixo.imshow(imagem)
        eixo.set_title(titulo)
        eixo.axis("off")

    plt.tight_layout()
    plt.show()

    print("ROI:", roi)
    print(
        "Diferença média absoluta entre os métodos:",
        float(np.mean(diferenca)),
    )
    print(
        "Maior diferença entre os métodos:",
        int(np.max(diferenca)),
    )

    return {
        "original": original,
        "undistort": corrigida_undistort,
        "remap": corrigida_remap,
        "diferenca": diferenca,
        "roi": roi,
    }

imagem_cam1 = primeiro_existente([
    "cam1_imagem_colorida.jpg",
    "imagem_original_calibracao.jpg",
])

imagem_cam2 = primeiro_existente([
    "cam2_imagem_colorida.jpg",
    "cam2_imagem_original_calibracao.jpg",
    "Capturas_2/cam2_frm0.jpg",
])

resultado_3D_cam1 = corrigir_distorcao(
    "Câmera 1",
    "parametros_calibracao.npz",
    imagem_cam1,
    "cam1_3D",
    alpha=1.0,
)

resultado_3D_cam2 = corrigir_distorcao(
    "Câmera 2",
    "cam2_parametros_calibracao.npz",
    imagem_cam2,
    "cam2_3D",
    alpha=1.0,
)

### 9.1 Resultado disponível da câmera 1

A correção da imagem colorida da primeira câmera foi executada durante a preparação deste relatório.

- Diferença média absoluta entre `undistort` e `remap`: **0.000044**
- Maior diferença observada entre os métodos: **1 níveis de intensidade**

Os dois métodos produziram imagens praticamente equivalentes. Pequenas diferenças são esperadas devido à interpolação e ao arredondamento numérico.

### 9.2 Análise visual da correção

As mudanças devem ser avaliadas principalmente nas bordas da imagem, onde a distorção radial tende a ser mais perceptível. Linhas retas próximas às extremidades devem apresentar menor curvatura após a correção.

`undistort()` executa diretamente a transformação. O método baseado em `remap()` calcula mapas de correspondência que podem ser reutilizados para várias imagens ou quadros de vídeo, sendo vantajoso em aplicações em tempo real.

O parâmetro `alpha` de `getOptimalNewCameraMatrix()` controla o compromisso entre preservar o campo de visão e eliminar regiões sem informação. Com `alpha=1`, preserva-se mais da imagem, podendo surgir bordas vazias. Com `alpha=0`, a imagem tende a ser mais recortada.

## 10. Análise e discussão

A calibração da primeira câmera apresentou resultados mais consistentes, com menor RMS e menor erro de reprojeção. A proximidade entre \(f_x\) e \(f_y\) nas duas câmeras indica que o modelo estimou pixels com proporções próximas de quadradas. O skew foi nulo em ambos os casos.

As diferenças entre os pontos principais podem ser explicadas por diferenças de montagem da lente, recorte do sensor, alinhamento óptico e enquadramento interno. Os coeficientes de distorção também são específicos de cada dispositivo.

A comparação das correções reforça que os parâmetros de uma câmera não devem ser aplicados a imagens de outra câmera. Além disso, as configurações de resolução e zoom precisam permanecer consistentes com aquelas utilizadas durante a calibração.

## 11. Conclusões

O experimento permitiu estimar os parâmetros intrínsecos, extrínsecos e de distorção de duas câmeras. A primeira câmera apresentou melhor qualidade de ajuste, enquanto a segunda exibiu erros maiores e coeficientes radiais de maior magnitude.

A existência de um conjunto \(R,t\) por imagem foi associada corretamente à mudança de pose do alvo em relação à câmera. A matriz \(K\) e o vetor de distorção permaneceram comuns às imagens de uma mesma câmera.

A correção de distorção foi implementada com os dois métodos solicitados. `undistort()` e `remap()` produziram resultados equivalentes, enquanto `remap()` oferece a vantagem de permitir a reutilização dos mapas em sequências de vídeo.

Portanto, os objetivos principais do laboratório foram atingidos: compreensão dos parâmetros de câmera, calibração de dispositivos distintos, análise dos erros e aplicação dos parâmetros na correção de imagens.

## 12. Referências

1. OpenCV. *Camera Calibration*. Disponível em: <https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html>.
2. LearnOpenCV. *Geometry of Image Formation*. Disponível em: <https://learnopencv.com/geometry-of-image-formation/>.
3. LearnOpenCV. *Camera Calibration using OpenCV*. Disponível em: <https://learnopencv.com/camera-calibration-using-opencv/>.
4. SZELISKI, Richard. *Computer Vision: Algorithms and Applications*. 2. ed. Springer, 2022.
5. KURASHIMA, Celso Setsuo. *Laboratório 4 — Calibração de Câmeras*. ESZA019, UFABC, 2026.

## 13. Checklist antes da publicação

- [ ] Preencher os nomes completos dos autores.
- [ ] Colocar as imagens de exemplo do professor em `Amostras_Professor/` e executar a Parte 2A.
- [ ] Copiar uma imagem colorida ou de calibração da câmera 2 para a pasta do relatório.
- [ ] Executar todas as células do notebook.
- [ ] Conferir se as imagens originais e corrigidas aparecem lado a lado.
- [ ] Salvar o notebook com as saídas.
- [ ] Publicar a pasta completa no GitHub.
- [ ] Adicionar o link do relatório ao índice HTML da equipe.